## 1. Setup



In [ ]:
# Enable IPython extension for automatically reloading modules before executing code
%load_ext autoreload
%autoreload 2

import os
from src.data_loader import load_raw_data
from src.visualization.exploratory import (
    plot_geographic_hotspots,
    plot_null_pattern_by_year,
    plot_target_distribution_by_vessel
)

print("Environment successfully initialized with dynamic autoreload.")

## 2. Load Data

In [ ]:
# Define the expected path for the dataset downloaded by the infrastructure layer
DATA_PATH = os.path.join("data", "raw", "piracy_attacks.csv")

# Load and optimize types via our functional data loader
df_raw = load_raw_data(DATA_PATH)

# Display structural telemetry of the dataset
print(f"Dataset loaded with {df_raw.shape[0]} records and {df_raw.shape[1]} columns.\n")
df_raw.info()

## 3. Exploratory Data Analysis (EDA)

In [ ]:

fig_nulls = plot_null_pattern_by_year(df_raw)
fig_nulls.show()

In [ ]:
fig_vessels = plot_target_distribution_by_vessel(df_raw, top_n=10)
fig_vessels.show()

In [ ]:
from src.visualization.exploratory import plot_geospatial_heatmap, plot_shore_distance_distribution
fig_distance = plot_shore_distance_distribution(df_raw)
fig_distance.show()

In [ ]:
from src.visualization.exploratory import plot_multidimensional_piracy_profile, plot_operational_status_matrix

# 1. Matriz de Probabilidade Tática (Muito superior a um gráfico de barras comum para cruzamento)
fig_matrix = plot_operational_status_matrix(df_raw)
fig_matrix.show()

# 2. O Fluxo de Coordenadas Paralelas (Visualização dinâmica avançada)
fig_flows = plot_multidimensional_piracy_profile(df_raw)
fig_flows.show()

In [ ]:


# 1. Mapa de Calor Geográfico Puro (Tira os pins e mostra as manchas de calor no mar)
geo_heatmap = plot_geospatial_heatmap(df_raw)
geo_heatmap

In [ ]:
piracy_map = plot_geographic_hotspots(df_raw)
piracy_map

## 4. Data Cleaning

In [ ]:
from src.features.transactions import tx_skewed_distance_transform, tx_nlp_vessel_mining, tx_clean_vessels, tx_handle_spatial_metadata, tx_spatial_binning, tx_discretize_time_gaps, tx_temporal_engineering, tx_drop_redundant_features


df_processed = (
    df_raw
    .pipe(tx_clean_vessels)              # 1. Limpeza estrutural de metadados
    .pipe(tx_nlp_vessel_mining)          # 2. NLP reconstrói vessel_type histórico (Nome + Descrição)
    .pipe(tx_handle_spatial_metadata)    # 4. Trata nulos de ZEE e cria is_international_waters
    .pipe(tx_spatial_binning)            # 5. Agrupa lat/lon em bins regionais
    .pipe(tx_skewed_distance_transform)  # 6. Aplica Log1p na distância e captura feat_is_near_port
    .pipe(tx_discretize_time_gaps)       # 7. Captura o comportamento de nulos do horário
    .pipe(tx_temporal_engineering)       # 8. Cria o âncoramento de ano e as ondas senoidais de Monções
    .pipe(tx_drop_redundant_features)
)

print(f"Dataset loaded with {df_processed.shape[0]} records and {df_processed.shape[1]} columns.\n")
df_processed.info()


In [ ]:
import plotly.express as px
from src.features.selection import compute_numerical_correlation, compute_categorical_nmi_matrix

# 1. Calcular e exibir a Correlação das Numéricas
df_corr = compute_numerical_correlation(df_processed)
fig_corr = px.imshow(
    df_corr, 
    text_auto=".2f", 
    title="Numerical Correlation Matrix (Pearson)",
    color_continuous_scale="RdBu", 
    zmin=-1.0,  
    zmax=1.0   
)
fig_corr.show()

# 2. Calcular e exibir o NMI das Categóricas (O poder preditivo das categorias)
df_nmi = compute_categorical_nmi_matrix(df_processed)
fig_nmi = px.imshow(
    df_nmi, 
    text_auto=".2f", 
    title="Categorical Normalized Mutual Information Matrix (NMI)",
    color_continuous_scale="Blues"
)
fig_nmi.show()

In [ ]:
import plotly.express as px
from src.features.selection import compute_target_association

# Calcular a associação de todas as colunas contra o attack_type
df_target_risk = compute_target_association(df_processed, target_col='attack_type')

# Plotar o Ranking de Força Preditiva
fig_target = px.bar(
    df_target_risk,
    x="NMI_with_Target",
    y="Feature",
    orientation="h",  # Gráfico horizontal para facilitar a leitura dos nomes das colunas
    title="Predictive Power Ranking (NMI) strictly against 'attack_type'",
    labels={"NMI_with_Target": "Normalized Mutual Information (Score)", "Feature": "Engineered Resource"},
    color="NMI_with_Target",
    color_continuous_scale="Blues",
    template="plotly_white"
)

# Força o plot a ordenar de cima para baixo (do maior NMI para o menor)
fig_target.update_layout(yaxis={'categoryorder':'total ascending'}, coloraxis_showscale=False)
fig_target.show()

## 5. Models and Evaluation


In [ ]:
from src.models.baseline import run_raw_baseline

# Run the estimators directly on the unmodified dataframe layout
df_raw_baseline = run_raw_baseline(df_raw)

print("Raw Baseline Performance (No Cleaning / No NLP / No Scalers):")
df_raw_baseline.sort_values(by="AUC_ROC_OVR", ascending=False)

In [ ]:
from src.models.train import run_temporal_benchmark

df_model_results = run_temporal_benchmark(df_processed)
df_model_results.sort_values(by="F1_Macro", ascending=False)